# Flood Mapping Data Pipeline (Colab)

GEOG761 Group 05 — Flood Mapping & Response

Fetches the six input layers from the project's architecture diagram and
saves them under `/content/data/raw/...`, ready for the ML model step:

| Layer | Source | Format |
|---|---|---|
| Land Type | ESA WorldCover v200 (Earth Engine) | GeoTIFF |
| Elevation | Copernicus GLO-30 DEM (Earth Engine) | GeoTIFF |
| SAR Satellite | Sentinel-1 GRD (Earth Engine) | GeoTIFF (VV/VH composite) |
| Population Density | WorldPop (Earth Engine) | GeoTIFF |
| Critical Infrastructure | OpenStreetMap (LINZ swap-in ready) | GeoJSON per category |
| Transport Infrastructure | OpenStreetMap (LINZ swap-in ready) | GeoJSON per category |

Run the cells top to bottom. The Earth Engine cell below opens a one-time
browser login the first time you run this notebook in a session.

**Optional: save outputs to Google Drive** so they survive when the Colab
runtime resets — see the "Mount Google Drive (optional)" cell.


## 1. Setup

In [ ]:
# Runtime packages. earthengine-api, requests, scikit-learn and matplotlib
# ship with Colab; the rest don't. geedim is geemap's tiled-download engine
# (see the `export_ee_image` helper in section 3) -- without it, exports
# over an area as large as the whole country hit Earth Engine's single-
# request pixel-grid limit and fail.
!pip install -q geemap osmnx geopandas python-dotenv rasterio geedim


In [ ]:
import ee

# One-time interactive login per Colab session/account. If this cell has
# already been authorised for this account, it returns immediately.
ee.Authenticate()

# Use your group's GEE-registered Cloud project.
EE_PROJECT = "ayao859-geog761"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready, project:", EE_PROJECT)


## 2. Mount Google Drive (optional)

Skip this cell if you're fine with outputs disappearing when the Colab
runtime resets. Otherwise mount Drive and outputs will be written under
`/content/drive/MyDrive/save-kiwis-from-flooding/data/raw/...` instead of
`/content/data/raw/...`.


In [ ]:
USE_DRIVE = False  # flip to True to persist outputs to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 3. Configuration

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import os

# --- Output paths ------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/save-kiwis-from-flooding") if USE_DRIVE else Path("/content")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"  # ML model + depth + figure outputs (sections 11-13)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RASTER_DIRS = {
    "land_type": RAW_DIR / "land_type",
    "elevation": RAW_DIR / "elevation",
    "sar": RAW_DIR / "sar",
    "population_density": RAW_DIR / "population_density",
}
VECTOR_DIRS = {
    "critical_infrastructure": RAW_DIR / "critical_infrastructure",
    "transport_infrastructure": RAW_DIR / "transport_infrastructure",
}
for _dir in (*RASTER_DIRS.values(), *VECTOR_DIRS.values()):
    _dir.mkdir(parents=True, exist_ok=True)

# --- Area of interest: whole of New Zealand -----------------------------------
COUNTRY_NAME = "New Zealand"
GEE_COUNTRY_TABLE = "USDOS/LSIB_SIMPLE/2017"
GEE_COUNTRY_NAME_FIELD = "country_na"

countries = ee.FeatureCollection(GEE_COUNTRY_TABLE)
NZ_BOUNDARY = countries.filter(ee.Filter.eq(GEE_COUNTRY_NAME_FIELD, COUNTRY_NAME)).geometry()

# NZ's bounding box, split into a grid of cells for chunking OSM/Overpass
# queries (see section 8 for why a plain geometric grid is used instead of
# named regions).
NZ_BBOX = (166.3, -47.5, 178.7, -34.0)  # (west, south, east, north)

# --- Event window --------------------------------------------------------------
# Sentinel-1 revisit over NZ is ~12 days; default to a rolling 24-day window.
# Set these explicitly to target a specific cyclone/flood event instead.
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=24)
# START_DATE, END_DATE = date(2026, 2, 10), date(2026, 2, 20)  # example override

# --- LINZ Data Service (optional) ----------------------------------------------
# Set LINZ_API_KEY to fetch Critical/Transport Infrastructure from LINZ
# instead of the OpenStreetMap fallback. Free key: sign in at
# https://data.linz.govt.nz -> profile menu -> "API Keys" -> "Generate API
# key", enabling the "Web Feature Service (WFS)" scope.
LINZ_API_KEY = os.environ.get("LINZ_API_KEY")  # or paste directly for a quick test
LINZ_WFS_BASE = "https://data.linz.govt.nz/services"

# --- Resolution ------------------------------------------------------------
# Country-wide exports default to a coarser scale to keep exports tractable;
# lower this for a smaller AOI / higher-resolution local test.
SCALE_M = 100

# --- Export helper -----------------------------------------------------------
# `geemap.ee_export_image` (a single getDownloadURL request) caps out at a
# ~32768-pixel grid per side -- fine for a small test AOI, but a whole-country
# image blows straight through it ("Pixel grid dimensions ... must be less
# than or equal to 32768"), and geemap prints a warning and moves on rather
# than raising, so a failed export can silently leave a missing/incomplete
# file. `geemap.download_ee_image` (backed by the geedim package installed
# above) tiles large requests into many small ones automatically, so it
# works at any AOI size -- use it for every raster export in this notebook.
import geemap

def export_ee_image(image, path, aoi, scale=SCALE_M, crs="EPSG:4326"):
    geemap.download_ee_image(image, filename=path, region=aoi, scale=scale, crs=crs)
    return path

print("AOI: whole of", COUNTRY_NAME)
print("SAR window:", START_DATE, "to", END_DATE)
print("Outputs under:", RAW_DIR)


## 4. Land Type — ESA WorldCover v200

10 m resolution, 11 land-cover classes (Sentinel-1/2 derived). Matches the
architecture diagram's "Land Type" box directly — no source substitution
needed.


In [ ]:
WORLDCOVER_COLLECTION = "ESA/WorldCover/v200"
LAND_TYPE_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}

def fetch_land_cover(aoi):
    image = ee.ImageCollection(WORLDCOVER_COLLECTION).first()
    return image.select("Map").clip(aoi).rename("land_type")

land_type_image = fetch_land_cover(NZ_BOUNDARY)
land_type_path = str(RASTER_DIRS["land_type"] / "nz_land_type.tif")
export_ee_image(land_type_image, land_type_path, NZ_BOUNDARY)
print("Saved:", land_type_path)


## 5. Elevation

Target source per the architecture diagram is the LINZ 8 m DEM (used for
the "bathtub model": ground elevation − SAR flood-extent boundary → water
surface elevation → depth). LINZ doesn't publish that DEM through Earth
Engine or a queryable API — it needs a manual export from
data.linz.govt.nz. This notebook defaults to the Copernicus GLO-30 DEM
(30 m, global, free via Earth Engine) and leaves a swap-in cell below.


In [ ]:
COPERNICUS_DEM_COLLECTION = "COPERNICUS/DEM/GLO30"

def fetch_elevation(aoi):
    dem = ee.ImageCollection(COPERNICUS_DEM_COLLECTION).select("DEM").mosaic()
    return dem.clip(aoi).rename("elevation_m")

elevation_image = fetch_elevation(NZ_BOUNDARY)
elevation_path = str(RASTER_DIRS["elevation"] / "nz_elevation.tif")
export_ee_image(elevation_image, elevation_path, NZ_BOUNDARY)
print("Saved:", elevation_path)


In [ ]:
# Swap-in point once a LINZ 8 m DEM GeoTIFF has been downloaded locally
# (requires a LINZ API key + manual export from data.linz.govt.nz):
#
#   import rioxarray
#   linz_dem = rioxarray.open_rasterio("/content/nz_8m_dem.tif")
#
# Use `linz_dem` instead of `elevation_image` for higher-resolution depth
# modelling once you have it.


## 6. SAR Satellite — Sentinel-1 GRD

Chosen per the group's satellite comparison (free/open access, 10 m
resolution, ~12-day revisit over NZ, used in operational systems like
Copernicus Global Flood Monitoring). Includes a baseline Otsu-style VV
threshold water mask as a sanity check against the ML model's predictions.


In [ ]:
S1_COLLECTION = "COPERNICUS/S1_GRD"

def fetch_sar_collection(aoi, start_date, end_date, orbit_pass=None):
    collection = (
        ee.ImageCollection(S1_COLLECTION)
        .filterBounds(aoi)
        .filterDate(str(start_date), str(end_date))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
    )
    if orbit_pass:
        collection = collection.filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))
    return collection

def to_water_mask(image, vv_threshold_db=-17.0):
    """Baseline water mask -- a placeholder for the real Otsu/adaptive
    threshold the ML model will compute, kept here as a sanity check."""
    return image.select("VV").lt(vv_threshold_db).rename("water_mask")

sar_collection = fetch_sar_collection(NZ_BOUNDARY, START_DATE, END_DATE)
sar_composite = sar_collection.select(["VV", "VH"]).median().clip(NZ_BOUNDARY)

sar_path = str(RASTER_DIRS["sar"] / f"nz_sar_{START_DATE}_to_{END_DATE}.tif")
export_ee_image(sar_composite, sar_path, NZ_BOUNDARY)
print("Saved:", sar_path)
print("Scenes used:", sar_collection.size().getInfo())


## 7. Population Density — WorldPop

Architecture diagram target: WorldPop (NZ, ~100 m) or the Stats NZ Census
meshblock grid. WorldPop is used here since it's published directly as a
GEE ImageCollection; the Stats NZ grid would need a separate download +
join step.


In [ ]:
WORLDPOP_COLLECTION = "WorldPop/GP/100m/pop"
WORLDPOP_YEAR = 2020

def fetch_population(aoi, year=WORLDPOP_YEAR):
    collection = (
        ee.ImageCollection(WORLDPOP_COLLECTION)
        .filter(ee.Filter.eq("country", "NZL"))
        .filter(ee.Filter.eq("year", year))
    )
    return collection.mosaic().clip(aoi).rename("population_density")

population_image = fetch_population(NZ_BOUNDARY)
population_path = str(RASTER_DIRS["population_density"] / f"nz_population_{WORLDPOP_YEAR}.tif")
export_ee_image(population_image, population_path, NZ_BOUNDARY)
print("Saved:", population_path)


## 8. Critical Infrastructure

Architecture diagram target sources: LINZ + Stats NZ / Civil Defence NZ,
which need a LINZ API key (see the LINZ section below). Until a key is
available, each category is fetched from OpenStreetMap instead — open
access, nationwide coverage, and a reasonable tag-for-tag match.

**Why a geometric grid instead of named regions:** the first version of
this notebook chunked queries by `"<Region> Region, New Zealand"` name and
every single one failed, for two different reasons visible in that run's
log:

1. `ConnectionError ... Connection refused` from `overpass-api.de` — the
   free public Overpass instance was refusing/dropping connections
   (overload, not something in this code). Fixed below by rotating through
   several public Overpass mirrors and retrying with backoff.
2. `TypeError: Nominatim did not geocode query '<Region> Region, New
   Zealand' to a geometry of type (Multi)Polygon` for Waikato, Hawke's Bay,
   West Coast, Canterbury and Otago — Nominatim didn't resolve those exact
   strings to boundary polygons at all, so nothing was ever going to be
   fetched for them regardless of the network issue above.

Splitting NZ's bounding box into a plain lat/lon grid (`NZ_GRID_CELLS`,
built in section 3) sidesteps problem 2 entirely — no place-name geocoding
involved — and the endpoint rotation + retry below addresses problem 1.
**This cell can still take a few minutes.**


In [ ]:
import time
import geopandas as gpd
import pandas as pd
import osmnx as ox
from shapely.geometry import box

try:
    from osmnx._errors import InsufficientResponseError
except ImportError:
    class InsufficientResponseError(Exception):
        pass

# Public Overpass API mirrors -- rotate through these on connection
# failures/refusals. overpass-api.de (osmnx's default) is a shared,
# volunteer-run instance that regularly refuses connections under load;
# these mirrors run the same software against the same data.
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.fr/api/interpreter",
]

ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True
ox.settings.max_query_area_size = 5_000_000_000  # fewer automatic sub-divisions -> fewer requests per cell


def nz_grid_cells(nx=6, ny=6, bbox=NZ_BBOX):
    """Split NZ's bounding box into nx * ny rectangular cells for chunked
    Overpass queries. Cells over open ocean just come back empty -- cheap
    to query, no special-casing needed."""
    west, south, east, north = bbox
    dx, dy = (east - west) / nx, (north - south) / ny
    return [
        box(west + i * dx, south + j * dy, west + (i + 1) * dx, south + (j + 1) * dy)
        for i in range(nx) for j in range(ny)
    ]


NZ_GRID_CELLS = nz_grid_cells()


def _fetch_cell_with_retry(polygon, tags):
    last_exc = None
    for attempt, endpoint in enumerate(OVERPASS_ENDPOINTS):
        ox.settings.overpass_endpoint = endpoint
        try:
            return ox.features_from_polygon(polygon, tags)
        except InsufficientResponseError:
            return None  # a real "nothing here" response, not a network failure
        except Exception as exc:
            last_exc = exc
            time.sleep(2 * (attempt + 1))  # back off, then try the next mirror
    print(f"  [warn] cell failed on every Overpass mirror: {last_exc!r}")
    return None


CRITICAL_INFRA_TAGS = {
    "hospitals_medical": {"amenity": ["hospital", "clinic"]},
    "emergency_services": {"amenity": ["fire_station", "police"], "emergency": ["ambulance_station"]},
    "power_substations": {"power": ["substation", "plant"]},
    "water_treatment": {"man_made": ["wastewater_plant", "water_works"]},
    "telecommunications": {"man_made": ["communications_tower", "tower"]},
    "schools_shelters": {"amenity": ["school"]},
    "civil_defence_shelters": {"emergency": ["assembly_point", "shelter"]},
}

def fetch_tags_nationwide(tags, cells=None, request_delay_s=1.0):
    cells = cells if cells is not None else NZ_GRID_CELLS
    frames = []
    for idx, cell in enumerate(cells):
        gdf = _fetch_cell_with_retry(cell, tags)
        if gdf is not None and not gdf.empty:
            gdf = gdf.copy()
            gdf["grid_cell"] = idx
            frames.append(gdf)
        time.sleep(request_delay_s)
    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    # Adjacent grid cells share an edge, so a feature that straddles one can
    # be returned by both; dedupe on OSM's own id rather than geometry
    # (geometry objects aren't reliably hashable across shapely versions).
    combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)
    if "osmid" in combined.columns:
        combined = combined.drop_duplicates(subset="osmid")
    return combined

def fetch_critical_infrastructure(use_linz=None):
    if use_linz is None:
        use_linz = bool(LINZ_API_KEY)
    results = {}
    for category, tags in CRITICAL_INFRA_TAGS.items():
        gdf = None
        if use_linz:
            gdf = fetch_linz_layer(category)  # defined in the LINZ section below
        if gdf is None:
            print(f"  fetching '{category}' from OpenStreetMap ({len(NZ_GRID_CELLS)} grid cells)...")
            gdf = fetch_tags_nationwide(tags)
        gdf = gdf.copy()
        gdf["category"] = category
        results[category] = gdf
    return results

critical_infra = fetch_critical_infrastructure()
for category, gdf in critical_infra.items():
    out_path = VECTOR_DIRS["critical_infrastructure"] / f"{category}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"  {category}: {len(gdf)} features -> {out_path}")


## 9. Transport Infrastructure

Architecture diagram target sources: LINZ, Auckland Transport /
Christchurch City open data, and Waka Kotahi NZ Transport Agency — each its
own agency with its own access process. Defaults to OpenStreetMap for the
same reason as Critical Infrastructure above, and reuses the same
grid-based `fetch_tags_nationwide` from section 8 (run that cell first).


In [ ]:
TRANSPORT_TAGS = {
    "roads_highways": {"highway": True},
    "bridges": {"bridge": True},
    "railways": {"railway": ["rail", "light_rail"]},
    "airports_airstrips": {"aeroway": ["aerodrome", "runway"]},
    "ports_harbours": {"landuse": ["harbour"]},
    "tunnels": {"tunnel": True},
    "transit_stations": {"public_transport": ["station", "stop_position"]},
}

def fetch_transport_infrastructure(use_linz=None):
    if use_linz is None:
        use_linz = bool(LINZ_API_KEY)
    results = {}
    for category, tags in TRANSPORT_TAGS.items():
        gdf = None
        if use_linz:
            gdf = fetch_linz_layer(category)  # defined in the LINZ section below
        if gdf is None:
            print(f"  fetching '{category}' from OpenStreetMap ({len(NZ_GRID_CELLS)} grid cells)...")
            gdf = fetch_tags_nationwide(tags)
        gdf = gdf.copy()
        gdf["category"] = category
        results[category] = gdf
    return results

transport_infra = fetch_transport_infrastructure()
for category, gdf in transport_infra.items():
    out_path = VECTOR_DIRS["transport_infrastructure"] / f"{category}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"  {category}: {len(gdf)} features -> {out_path}")


## 10. Optional: switch Critical/Transport Infrastructure to LINZ

Skip this section entirely while `LINZ_API_KEY` is unset — sections 8 and 9
already ran on OpenStreetMap above. Come back to it once you have a key:

1. Register / sign in at https://data.linz.govt.nz
2. Profile menu -> **API Keys** -> **Generate API key**, enable the **Web
   Feature Service (WFS)** scope.
3. Set `LINZ_API_KEY` in the Configuration cell above (section 3).
4. Look up each dataset's numeric `layer-XXXXX` id on its LINZ data page and
   fill it into `LINZ_LAYER_IDS` below.
5. Re-run the Critical/Transport Infrastructure cells (8 and 9) — they call
   `fetch_linz_layer` automatically once `LINZ_API_KEY` is set.


In [ ]:
import requests

# Fill in with confirmed layer ids from https://data.linz.govt.nz once a key
# is available. Left as None so `fetch_linz_layer` fails loudly instead of
# silently querying the wrong layer.
LINZ_LAYER_IDS = {
    "roads_highways": None,   # e.g. NZ Roads (Topo)
    "railways": None,         # e.g. NZ Railway Centrelines (Topo)
    "hospitals_medical": None,
    # add the rest of the categories from sections 8/9 as you confirm their layer ids
}

def fetch_linz_layer(category, bbox=None):
    """Fetch a LINZ vector layer via WFS. Returns None (not an error) so the
    calling code can fall back to OpenStreetMap for any unconfigured category."""
    if not LINZ_API_KEY:
        return None
    layer_id = LINZ_LAYER_IDS.get(category)
    if layer_id is None:
        print(f"  [info] no LINZ layer id configured for '{category}' -- using OpenStreetMap")
        return None

    url = f"{LINZ_WFS_BASE};key={LINZ_API_KEY}/wfs"
    params = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "typeNames": f"layer-{layer_id}", "outputFormat": "json", "srsName": "EPSG:4326",
    }
    if bbox:
        params["bbox"] = ",".join(str(v) for v in bbox)
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    return gpd.GeoDataFrame.from_features(response.json()["features"], crs="EPSG:4326")


## 11. Machine Learning Model — Random Forest (W9)

Random Forest, per the group's plan and the pitch deck's satellite/ML
comparison (slide 7) -- the simplest model that still uses spatial/feature
context beyond a bare threshold. Same train/evaluate pattern as the
GEOG761 labs (Earth Engine `.sample()` -> pandas -> scikit-learn, e.g.
761_Lab3's cloud classifier): just swap `KMeans` for
`RandomForestClassifier`, and build the feature stack from land type /
elevation / SAR instead of Sentinel-2 bands.

There's no manually labelled flood extent to train on, so the baseline
Otsu-threshold water mask from section 6 (`to_water_mask`) is used as a
weak label to bootstrap training samples -- the same
bootstrap-from-a-simple-rule pattern the lab used for its cloud/clear/edge
sampling zones.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# --- Build the feature stack ------------------------------------------------
# Same three bands used for both the quick sklearn evaluation below and the
# scalable Earth Engine classification that follows it.
FEATURE_BANDS = ["VV", "VH", "elevation_m", "land_type"]

feature_stack = (
    sar_composite
    .addBands(elevation_image)
    .addBands(land_type_image)
    .select(FEATURE_BANDS)
)

# Weak labels: the section-6 Otsu-style threshold water mask.
water_mask = to_water_mask(sar_composite)  # 1 = water, 0 = not water
labelled_stack = feature_stack.addBands(water_mask.rename("class"))

# --- Sample points from each class, same fc_to_df pattern as the labs ------
def fc_to_df(fc):
    features = fc.getInfo()["features"]
    rows = []
    for f in features:
        props = f["properties"]
        coords = f["geometry"]["coordinates"]
        props["longitude"], props["latitude"] = coords[0], coords[1]
        rows.append(props)
    return pd.DataFrame(rows)

SAMPLES_PER_CLASS = 2000

water_samples = labelled_stack.updateMask(water_mask).sample(
    region=NZ_BOUNDARY, scale=SCALE_M, numPixels=SAMPLES_PER_CLASS, seed=0, geometries=True
)
dry_samples = labelled_stack.updateMask(water_mask.Not()).sample(
    region=NZ_BOUNDARY, scale=SCALE_M, numPixels=SAMPLES_PER_CLASS, seed=1, geometries=True
)

water_df = fc_to_df(water_samples)
dry_df = fc_to_df(dry_samples)
print(f"Water samples: {len(water_df)}, dry samples: {len(dry_df)}")


In [ ]:
# --- Quick client-side evaluation, same balanced-sampling + classification_report
# pattern as the lab's held-out test split. This is a sanity check, not the
# model used for the full-country map (see the next cell for why).
combined = pd.concat([water_df, dry_df]).dropna(subset=FEATURE_BANDS)
min_n = combined["class"].value_counts().min()
balanced = combined.groupby("class").sample(n=min_n, random_state=42)

X = balanced[FEATURE_BANDS].values
y = balanced["class"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Not water", "Water"]))
print("Feature importances:", dict(zip(FEATURE_BANDS, rf.feature_importances_.round(3))))


In [ ]:
# The sklearn model above is for quick evaluation only (same role as the
# lab's held-out classification_report) -- classifying every pixel across
# the whole country with it would mean pulling every pixel to Python one
# by one, which doesn't scale. Earth Engine's own Random Forest
# (`ee.Classifier.smileRandomForest`), trained on the same sampled points,
# runs server-side over the full image instead.
training_fc = water_samples.merge(dry_samples)

ee_rf_classifier = ee.Classifier.smileRandomForest(numberOfTrees=200, seed=42).train(
    features=training_fc, classProperty="class", inputProperties=FEATURE_BANDS
)

flood_extent = feature_stack.classify(ee_rf_classifier).rename("flood_extent")  # 1 = flooded

FLOOD_EXTENT_PATH = str(PROCESSED_DIR / "nz_flood_extent.tif")
export_ee_image(flood_extent, FLOOD_EXTENT_PATH, NZ_BOUNDARY)
print("Saved:", FLOOD_EXTENT_PATH)


## 12. Depth Calculation — "bathtub model" (W10)

Same method named in the architecture diagram (slide 6): ground elevation
(the Copernicus DEM from section 5) minus the SAR/Random-Forest flood
extent gives an estimated water-surface elevation, and depth is the
difference between that surface and the ground beneath it.

Implemented as a per-patch bathtub: within each connected flooded region,
the water surface is assumed flat and set to that patch's *highest* ground
elevation -- a conservative, simple version of the bathtub model, good
enough for a first depth estimate, not a hydraulic simulation.


In [ ]:
flood_mask = flood_extent.eq(1).selfMask()

# Label each connected flooded patch, then take the max ground elevation
# within each patch as that patch's (flat) water-surface elevation.
patches = flood_mask.connectedComponents(connectedness=ee.Kernel.plus(1), maxSize=1024)
water_surface = (
    elevation_image.updateMask(flood_mask)
    .addBands(patches.select("labels"))
    .reduceConnectedComponents(reducer=ee.Reducer.max(), labelBand="labels", maxSize=1024)
)

flood_depth = (
    water_surface.subtract(elevation_image)
    .max(0)
    .updateMask(flood_mask)
    .rename("depth_m")
)

DEPTH_PATH = str(PROCESSED_DIR / "nz_flood_depth.tif")
export_ee_image(flood_depth, DEPTH_PATH, NZ_BOUNDARY)
print("Saved:", DEPTH_PATH)


## 13. Outputs & Figures (W11)

Turns the flood extent + depth rasters into the two things the group
needs for the report/slides: which Critical/Transport Infrastructure
features fall inside the flood extent, and a couple of figures to drop
straight into the deck.


In [ ]:
import rasterio
from rasterio.features import shapes as raster_shapes
from shapely.geometry import shape as shapely_shape

# Vectorise the flood extent raster into polygons once; everything else
# below is an ordinary geopandas spatial check against that polygon set.
with rasterio.open(FLOOD_EXTENT_PATH) as src:
    flood_arr = src.read(1)
    flood_transform = src.transform
    flood_crs = src.crs

flood_polys = [
    shapely_shape(geom)
    for geom, value in raster_shapes(flood_arr, mask=(flood_arr == 1), transform=flood_transform)
]
flood_extent_gdf = gpd.GeoDataFrame(geometry=flood_polys, crs=flood_crs)
flood_union = flood_extent_gdf.unary_union
print(f"Flood extent: {len(flood_extent_gdf)} polygon(s)")


In [ ]:
def intersects_flood(geom):
    return geom is not None and flood_union.intersects(geom)

def affected_summary(layers):
    rows = []
    for category, gdf in layers.items():
        if gdf.empty:
            rows.append({"category": category, "total": 0, "affected": 0})
            continue
        gdf = gdf.to_crs(flood_extent_gdf.crs) if gdf.crs else gdf.set_crs(flood_extent_gdf.crs)
        affected = gdf.geometry.apply(intersects_flood).sum()
        rows.append({"category": category, "total": len(gdf), "affected": int(affected)})
    return pd.DataFrame(rows)

critical_summary = affected_summary(critical_infra)
transport_summary = affected_summary(transport_infra)

print("Critical infrastructure:\n", critical_summary)
print("\nTransport infrastructure:\n", transport_summary)

critical_summary.to_csv(PROCESSED_DIR / "critical_infrastructure_affected.csv", index=False)
transport_summary.to_csv(PROCESSED_DIR / "transport_infrastructure_affected.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, summary, title in zip(
    axes, [critical_summary, transport_summary], ["Critical Infrastructure", "Transport Infrastructure"]
):
    ax.bar(summary["category"], summary["total"], label="Total", alpha=0.4)
    ax.bar(summary["category"], summary["affected"], label="Affected by flood", alpha=0.9)
    ax.set_xticklabels(summary["category"], rotation=45, ha="right")
    ax.set_title(title)
    ax.legend()
fig.tight_layout()

bar_fig_path = str(PROCESSED_DIR / "affected_infrastructure.png")
fig.savefig(bar_fig_path, dpi=150)
print("Saved:", bar_fig_path)
plt.show()


In [ ]:
with rasterio.open(DEPTH_PATH) as src:
    depth_arr = src.read(1, masked=True)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(depth_arr, cmap="Blues")
ax.set_title("Estimated flood depth (m)")
ax.axis("off")
fig.colorbar(im, ax=ax, label="Depth (m)", shrink=0.7)

depth_fig_path = str(PROCESSED_DIR / "flood_depth_map.png")
fig.savefig(depth_fig_path, dpi=150)
print("Saved:", depth_fig_path)
plt.show()


## 14. Summary

All six input layers plus the Random Forest flood-extent map, depth
raster, affected-infrastructure tables and figures are now under
`data/raw/` and `data/processed/` (or your mounted Drive folder if
`USE_DRIVE = True`), ready to drop into the W11 slides.


In [ ]:
for name, path in {**RASTER_DIRS, **VECTOR_DIRS, "processed": PROCESSED_DIR}.items():
    files = sorted(p.name for p in path.glob("*"))
    print(f"{name}: {files}")
